# Factory definition

M5 gives us real demand, product identity, category, and price. It does **not** give us a manufacturing system: machines, routings, processing/setup times, production costs, materials, or inventory policy . This notebook defines that fictional factory around the 10 products selected, and writes the resulting tables to `data/optimization/`:

- `products.csv` : per-product economics
- `machines.csv` : the 6 machines
- `routing.csv` : which machine(s) can make which product, at what speed/cost
- `materials.csv` + `bill_of_materials.csv` : raw materials and consumption per unit
- `machine_availability.csv` : weekly capacity baseline over the 12-week planning horizon

**Every number below is tagged as either REAL (from M5) or ASSUMED (an explicit, documented industrial-engineering assumption)**, see the summary table at the end. Nothing is presented as observed data unless it actually came from `demand.csv` or `sell_prices.csv`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("../data/optimization")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Real data inputs

Pull the only two things the factory model is grounded in from real data: average/std **weekly
demand** per product (from `demand.csv` -- already free of the partial final week) and average
**selling price** per product (from `sell_prices.csv`, M5).

In [2]:
demand = pd.read_csv("../data/processed/demand/demand.csv", parse_dates=["date"])
price = pd.read_csv("../data/raw/m5/sell_prices.csv")

products_meta = (
    demand[["product_id", "m5_item_id", "category"]]
    .drop_duplicates()
    .sort_values("product_id")
    .reset_index(drop=True)
)

# demand.csv (notebook 02) already drops the one partial week, so every week here is a
# full 7-day week -- no further filtering needed.
weekly = demand.groupby(["product_id", "week"])["demand"].sum().reset_index()

demand_stats = (
    weekly.groupby("product_id")["demand"]
    .agg(mean_weekly="mean", std_weekly="std")
    .round(2)
)

price_stats = price.groupby("item_id")["sell_price"].mean().round(2).rename("selling_price")

real_inputs = (
    products_meta
    .merge(demand_stats, on="product_id")
    .merge(price_stats, left_on="m5_item_id", right_index=True)
)
real_inputs

,product_id,m5_item_id,category,mean_weekly,std_weekly,selling_price
0,P01,FOODS_3_586,FOODS,3367.46,504.43,1.60
1,P02,FOODS_1_218,FOODS,858.44,357.62,0.98
2,P03,FOODS_3_395,FOODS,56.29,15.08,3.96
3,P04,FOODS_2_076,FOODS,20.45,15.28,10.10
4,P05,HOUSEHOLD_1_083,HOUSEHOLD,468.77,135.65,0.98
5,P06,HOUSEHOLD_1_176,HOUSEHOLD,56.05,16.62,3.89
6,P07,HOUSEHOLD_2_001,HOUSEHOLD,20.52,8.21,6.59
7,P08,HOBBIES_1_371,HOBBIES,448.62,186.01,0.48
8,P09,HOBBIES_1_151,HOBBIES,46.67,13.71,5.15
9,P10,HOBBIES_1_225,HOBBIES,19.22,12.70,29.67


## 1. Products

`selling_price` is real (M5 average sell price). Everything else is an **assumed** formula,
applied uniformly across all 10 products so the logic is transparent and auditable:

| Field | Formula | Assumption |
|---|---|---|
| `production_cost` | `selling_price / 1.35` | flat 35% gross margin |
| `holding_cost` | `production_cost × 2%` | 2% of unit cost per week (~annualized cost of capital + storage) |
| `shortage_cost` | `selling_price × 1.5` | lost sale costs 1.5× the margin, lost revenue + service penalty |
| `safety_stock` | `1.0 × std_weekly` | 1 week lead time, z = 1.0 (≈ 84% cycle service level) |
| `max_inventory` | `4 × mean_weekly` | 4 weeks of average demand as a storage cap |

These rates are deliberately simple and identical across products, the point is to have a
traceable, documented rule rather than hand-picked per-product numbers.

In [3]:
MARGIN = 0.35
HOLDING_RATE = 0.02
SHORTAGE_MULTIPLIER = 1.5
SAFETY_STOCK_Z = 1.0
MAX_INVENTORY_WEEKS = 4

products = real_inputs.copy()
products["production_cost"] = (products["selling_price"] / (1 + MARGIN)).round(2)
products["holding_cost"] = (products["production_cost"] * HOLDING_RATE).round(3)
products["shortage_cost"] = (products["selling_price"] * SHORTAGE_MULTIPLIER).round(2)
products["safety_stock"] = (SAFETY_STOCK_Z * products["std_weekly"]).round(0).astype(int)
products["max_inventory"] = (MAX_INVENTORY_WEEKS * products["mean_weekly"]).round(0).astype(int)

products = products[["product_id", "m5_item_id", "category", "production_cost", "selling_price","holding_cost", "shortage_cost", "safety_stock", "max_inventory"]]
products

,product_id,m5_item_id,category,production_cost,selling_price,holding_cost,shortage_cost,safety_stock,max_inventory
0,P01,FOODS_3_586,FOODS,1.19,1.60,0.024,2.40,504,13470
1,P02,FOODS_1_218,FOODS,0.73,0.98,0.015,1.47,358,3434
2,P03,FOODS_3_395,FOODS,2.93,3.96,0.059,5.94,15,225
3,P04,FOODS_2_076,FOODS,7.48,10.10,0.150,15.15,15,82
4,P05,HOUSEHOLD_1_083,HOUSEHOLD,0.73,0.98,0.015,1.47,136,1875
5,P06,HOUSEHOLD_1_176,HOUSEHOLD,2.88,3.89,0.058,5.84,17,224
6,P07,HOUSEHOLD_2_001,HOUSEHOLD,4.88,6.59,0.098,9.88,8,82
7,P08,HOBBIES_1_371,HOBBIES,0.36,0.48,0.007,0.72,186,1794
8,P09,HOBBIES_1_151,HOBBIES,3.81,5.15,0.076,7.72,14,187
9,P10,HOBBIES_1_225,HOBBIES,21.98,29.67,0.440,44.50,13,77


In [4]:
products.to_csv(OUTPUT_DIR / "products.csv", index=False)
print("Saved:", (OUTPUT_DIR / "products.csv").resolve())

Saved: C:\Users\HP\Walid\Industrial_production_optimization\data\optimization\products.csv


## 2. Machines

6 machines (within the spec's 5–8 range), organized in three tiers so the factory has genuine bottlenecks and genuine slack, not every machine equally busy:

- **M01, M02 : high-throughput lines.** Dedicated/near-dedicated to the two highest-volume FOODS products (P01, P02). Longer regular hours (extra shift), shorter setup times (optimized for fast changeover), more labor.
- **M03, M04 : general-purpose machines.** Handle the high-volume HOUSEHOLD/HOBBIES products (P05, P08) plus the medium-volume products (P03, P06). Standard hours.
- **M05, M06 : specialty/low-volume machines.** Handle the low-volume, longer-setup products
  (P04, P07, P09, P10). Shorter regular hours (run less continuously), less labor, but longer setup times (precision/small-batch work).

In [5]:
machines = pd.DataFrame(
    [
        # machine_id, regular_hours, overtime_hours, labor_requirement, maintenance_cost, availability
        ("M01", 90, 18, 3, 1200, 0.97),
        ("M02", 80, 16, 2, 900, 0.95),
        ("M03", 80, 16, 2, 900, 0.93),
        ("M04", 80, 16, 2, 850, 0.93),
        ("M05", 70, 10, 1, 600, 0.90),
        ("M06", 70, 10, 1, 550, 0.90),
    ],
    columns=[
        "machine_id", "regular_hours", "overtime_hours", "labor_requirement",
        "maintenance_cost", "availability",
    ],
)
machines

,machine_id,regular_hours,overtime_hours,labor_requirement,maintenance_cost,availability
0,M01,90,18,3,1200,0.97
1,M02,80,16,2,900,0.95
2,M03,80,16,2,900,0.93
3,M04,80,16,2,850,0.93
4,M05,70,10,1,600,0.90
5,M06,70,10,1,550,0.90


In [6]:
machines.to_csv(OUTPUT_DIR / "machines.csv", index=False)
print("Saved:", (OUTPUT_DIR / "machines.csv").resolve())

Saved: C:\Users\HP\Walid\Industrial_production_optimization\data\optimization\machines.csv


## 3. Routing

Every product gets **two** candidate machines (a primary and a slower/alternate secondary),
matching the spec's routing example. Assignment follows the machine tiers above; a product's
*primary* processing time is set so that, at its real average weekly demand, the primary
machine's utilization from that product alone lands roughly in the 25–80% range, tight enough
on M01/M03/M04 to make overtime/alternate-routing decisions matter, slack enough on M02/M05/M06
to give the optimizer room to shift load. The *secondary* route is deliberately slower
(unfamiliar machine, more manual setup) and costs more time per unit, a real overflow option, not a free alternative.

Setup times scale with machine tier (fast lines: 1.0–1.5h; general-purpose: 1.5–2.0h; specialty:
2.0–2.5h). `setup_cost = setup_time × $60/h` (an assumed blended shop rate covering labor +
downtime). `min_batch_size` is tiered by volume (high: 150, medium: 40, low: 15 units) so setups
aren't trivially amortized over a handful of units.

In [7]:
SETUP_COST_RATE = 60  # $ per setup-hour

MIN_BATCH_BY_TIER = {"high": 150, "medium": 40, "low": 15}
PRODUCT_TIER = {
    "P01": "high", "P02": "high", "P05": "high", "P08": "high",
    "P03": "medium", "P06": "medium", "P09": "medium",
    "P04": "low", "P07": "low", "P10": "low",
}

# product_id, machine_id, processing_time (h/unit), setup_time (h), role
routing_raw = [
    ("P01", "M01", 0.020, 1.0, "primary"),
    ("P01", "M02", 0.025, 1.5, "secondary"),
    ("P02", "M02", 0.040, 1.0, "primary"),
    ("P02", "M01", 0.060, 1.5, "secondary"),
    ("P05", "M03", 0.100, 1.5, "primary"),
    ("P05", "M02", 0.090, 2.0, "secondary"),
    ("P08", "M04", 0.100, 1.5, "primary"),
    ("P08", "M03", 0.120, 2.0, "secondary"),
    ("P03", "M03", 0.300, 1.5, "primary"),
    ("P03", "M04", 0.320, 2.0, "secondary"),
    ("P06", "M04", 0.280, 1.5, "primary"),
    ("P06", "M03", 0.300, 2.0, "secondary"),
    ("P09", "M05", 0.350, 2.0, "primary"),
    ("P09", "M06", 0.380, 2.5, "secondary"),
    ("P04", "M05", 0.400, 2.0, "primary"),
    ("P04", "M06", 0.420, 2.5, "secondary"),
    ("P07", "M06", 0.420, 2.0, "primary"),
    ("P07", "M05", 0.450, 2.5, "secondary"),
    ("P10", "M06", 0.500, 2.0, "primary"),
    ("P10", "M05", 0.550, 2.5, "secondary"),
]

routing = pd.DataFrame(
    routing_raw, columns=["product_id", "machine_id", "processing_time", "setup_time", "role"]
)
routing["setup_cost"] = (routing["setup_time"] * SETUP_COST_RATE).round(2)
routing["min_batch_size"] = routing["product_id"].map(PRODUCT_TIER).map(MIN_BATCH_BY_TIER)

routing_out = routing[
    ["product_id", "machine_id", "processing_time", "setup_time", "setup_cost", "min_batch_size"]
].copy()
routing_out

,product_id,machine_id,processing_time,setup_time,setup_cost,min_batch_size
0,P01,M01,0.020,1.0,60.0,150
1,P01,M02,0.025,1.5,90.0,150
2,P02,M02,0.040,1.0,60.0,150
3,P02,M01,0.060,1.5,90.0,150
4,P05,M03,0.100,1.5,90.0,150
5,P05,M02,0.090,2.0,120.0,150
6,P08,M04,0.100,1.5,90.0,150
7,P08,M03,0.120,2.0,120.0,150
8,P03,M03,0.300,1.5,90.0,40
9,P03,M04,0.320,2.0,120.0,40


**Sanity check** : weekly hours required to cover each machine's *primary* products at real
average demand, against `regular_hours`. This confirms the intended mix of tight and slack
machines rather than an accidentally infeasible or trivial factory.

In [8]:
mean_weekly = demand_stats["mean_weekly"]

primary_load = (
    routing[routing["role"] == "primary"]
    .assign(weekly_hours=lambda d: d["product_id"].map(mean_weekly) * d["processing_time"])
    .groupby("machine_id")["weekly_hours"]
    .sum()
)

utilization = (
    machines.set_index("machine_id")["regular_hours"]
    .to_frame()
    .join(primary_load.rename("primary_hours_needed"))
    .fillna(0)
)
utilization["utilization"] = (
    utilization["primary_hours_needed"] / utilization["regular_hours"]
).round(2)
utilization

,regular_hours,primary_hours_needed,utilization
machine_id,,,
M01,90,67.3492,0.75
M02,80,34.3376,0.43
M03,80,63.7640,0.80
M04,80,60.5560,0.76
M05,70,24.5145,0.35
M06,70,18.2284,0.26


In [9]:
routing_out.to_csv(OUTPUT_DIR / "routing.csv", index=False)
print("Saved:", (OUTPUT_DIR / "routing.csv").resolve())

Saved: C:\Users\HP\Walid\Industrial_production_optimization\data\optimization\routing.csv


## 4. Materials & bill of materials

- **One product-specific primary material per product** (RM01–RM10, matching P01–P10) : each
  product has its own core recipe/component, not a category-pooled one.
- **One shared packaging material** (`PKG01`) used by all 10 products : shared packaging lines / co-packing are common even across very different SKUs.
- **Two shared "cross-cutting" materials** that create genuine, non-category-siloed contention:
  - `SHR01` : a common plastic resin/component used by every HOUSEHOLD *and* HOBBIES product
    (both categories plausibly use similar molded/plastic components), so a shortage forces the optimizer to prioritize across category lines, not within one.
  - `SHR02` : a common bulk ingredient shared only by the two high-volume FOODS products (P01, P02), reflecting a bulk input two large-volume SKUs might be sourced together on, while the smaller/specialty FOODS items (P03, P04) source independently.

Each product's primary + shared-material quantities sum to per-tier totals (high: 0.9, medium: 1.0, low: 1.2 units per unit of product), packaging stays a flat 1.0 unit per unit of product.

`unit_cost` for each product-specific primary material is set to roughly half of that product's
`production_cost` (material ≈ half of unit cost, the rest labor/overhead).
`lead_time` is 1 week for high/medium-tier primary materials, 2 weeks for low-tier ones (small
specialty inputs, sourced less frequently); packaging and the two shared materials are treated as
externally-sourced, 2-week lead time. `availability` is sized to ~1.15–1.2× expected weekly
consumption at real average demand, same rule as before.

In [10]:
# Each product's own primary material, sized so primary + shared (where applicable) sums to
# the same per-tier total as the old single-material design (high 0.9, medium 1.0, low 1.2).
PRIMARY_MATERIAL = {f"P{str(i).zfill(2)}": f"RM{str(i).zfill(2)}" for i in range(1, 11)}

# product_id, material_id, quantity_required
bom_rows = [
    # FOODS high-volume pair shares a bulk ingredient (SHR02); specialty FOODS (P03, P04) don't
    ("P01", PRIMARY_MATERIAL["P01"], 0.6), ("P01", "SHR02", 0.3),
    ("P02", PRIMARY_MATERIAL["P02"], 0.6), ("P02", "SHR02", 0.3),
    ("P03", PRIMARY_MATERIAL["P03"], 1.0),
    ("P04", PRIMARY_MATERIAL["P04"], 1.2),
    # HOUSEHOLD + HOBBIES all share a common plastic resin/component (SHR01)
    ("P05", PRIMARY_MATERIAL["P05"], 0.6), ("P05", "SHR01", 0.3),
    ("P06", PRIMARY_MATERIAL["P06"], 0.7), ("P06", "SHR01", 0.3),
    ("P07", PRIMARY_MATERIAL["P07"], 0.8), ("P07", "SHR01", 0.4),
    ("P08", PRIMARY_MATERIAL["P08"], 0.6), ("P08", "SHR01", 0.3),
    ("P09", PRIMARY_MATERIAL["P09"], 0.7), ("P09", "SHR01", 0.3),
    ("P10", PRIMARY_MATERIAL["P10"], 0.8), ("P10", "SHR01", 0.4),
]

# Packaging: flat 1.0 unit per unit of product, for every product
bom_rows += [(product_id, "PKG01", 1.0) for product_id in PRIMARY_MATERIAL]

bill_of_materials = pd.DataFrame(bom_rows, columns=["product_id", "material_id", "quantity_required"])
bill_of_materials = bill_of_materials.sort_values(["product_id", "material_id"]).reset_index(drop=True)
bill_of_materials

,product_id,material_id,quantity_required
0,P01,PKG01,1.0
1,P01,RM01,0.6
2,P01,SHR02,0.3
3,P02,PKG01,1.0
4,P02,RM02,0.6
5,P02,SHR02,0.3
6,P03,PKG01,1.0
7,P03,RM03,1.0
8,P04,PKG01,1.0
9,P04,RM04,1.2


In [11]:
weekly_consumption = (
    bill_of_materials
    .assign(weekly_units=lambda d: d["product_id"].map(mean_weekly) * d["quantity_required"])
    .groupby("material_id")["weekly_units"]
    .sum()
)

production_cost_by_product = products.set_index("product_id")["production_cost"]

PRIMARY_MATERIAL_COST_SHARE = 0.5  # primary material assumed to be ~half of production_cost
LEAD_TIME_BY_TIER = {"high": 1, "medium": 1, "low": 2}
SHARED_UNIT_COST = {"PKG01": 0.15, "SHR01": 0.50, "SHR02": 0.55}
SHARED_LEAD_TIME = {"PKG01": 2, "SHR01": 2, "SHR02": 2}
SLACK_FACTOR = 1.20  # PKG01 slack is trimmed slightly below, see PACKAGING_SLACK_FACTOR
PACKAGING_SLACK_FACTOR = 1.15

materials_rows = []
for product_id, material_id in PRIMARY_MATERIAL.items():
    tier = PRODUCT_TIER[product_id]
    unit_cost = round(production_cost_by_product[product_id] * PRIMARY_MATERIAL_COST_SHARE, 3)
    materials_rows.append((material_id, unit_cost, LEAD_TIME_BY_TIER[tier]))

for material_id in ["PKG01", "SHR01", "SHR02"]:
    materials_rows.append((material_id, SHARED_UNIT_COST[material_id], SHARED_LEAD_TIME[material_id]))

materials = pd.DataFrame(materials_rows, columns=["material_id", "unit_cost", "lead_time"])
materials["availability"] = materials["material_id"].map(
    lambda m: round(
        weekly_consumption[m] * (PACKAGING_SLACK_FACTOR if m == "PKG01" else SLACK_FACTOR)
    )
)
materials = materials[["material_id", "availability", "unit_cost", "lead_time"]]
materials

,material_id,availability,unit_cost,lead_time
0,RM01,2425,0.595,1
1,RM02,618,0.365,1
2,RM03,68,1.465,1
3,RM04,29,3.740,2
4,RM05,338,0.365,1
5,RM06,47,1.440,1
6,RM07,20,2.440,2
7,RM08,323,0.180,1
8,RM09,39,1.905,1
9,RM10,18,10.990,2


In [12]:
materials.to_csv(OUTPUT_DIR / "materials.csv", index=False)
bill_of_materials.to_csv(OUTPUT_DIR / "bill_of_materials.csv", index=False)
print("Saved:", (OUTPUT_DIR / "materials.csv").resolve())
print("Saved:", (OUTPUT_DIR / "bill_of_materials.csv").resolve())

Saved: C:\Users\HP\Walid\Industrial_production_optimization\data\optimization\materials.csv
Saved: C:\Users\HP\Walid\Industrial_production_optimization\data\optimization\bill_of_materials.csv


## 5. Machine availability

A 12-week planning horizon. The baseline is simply each machine's `regular_hours` held constant every week, **no disruption is baked in here**. Machine breakdowns / maintenance dips are a deliberate experiment and will be produced by perturbing this file, not by hard-coding a failure into the baseline factory definition.

In [13]:
PLANNING_WEEKS = 12

machine_availability = pd.DataFrame(
    [
        (week, row.machine_id, row.regular_hours)
        for week in range(1, PLANNING_WEEKS + 1)
        for row in machines.itertuples()
    ],
    columns=["week", "machine_id", "available_hours"],
)
machine_availability.head(8)

,week,machine_id,available_hours
0,1,M01,90
1,1,M02,80
2,1,M03,80
3,1,M04,80
4,1,M05,70
5,1,M06,70
6,2,M01,90
7,2,M02,80


In [14]:
machine_availability.to_csv(OUTPUT_DIR / "machine_availability.csv", index=False)
print("Saved:", (OUTPUT_DIR / "machine_availability.csv").resolve())
print("Shape:", machine_availability.shape)

Saved: C:\Users\HP\Walid\Industrial_production_optimization\data\optimization\machine_availability.csv
Shape: (72, 3)


## Summary

| File | Real (M5) | Assumed |
|---|---|---|
| `products.csv` | `product_id`, `m5_item_id`, `category`, `selling_price` | `production_cost`, `holding_cost`, `shortage_cost`, `safety_stock`, `max_inventory` (all formula-derived, documented above) |
| `machines.csv` | — | entire table (machine count, hours, labor, maintenance, availability) |
| `routing.csv` | — (machine assignment informed by real `mean_weekly`/`std_weekly` demand, but the table itself is assumed) | entire table (processing time, setup time/cost, min batch size) |
| `materials.csv` | — (availability sized from real weekly demand; primary-material `unit_cost` derived from real `selling_price` via `production_cost`) | entire table (material list/structure, cost share formula, lead time) |
| `bill_of_materials.csv` | — | entire table (material assignment and quantities per unit) |
| `machine_availability.csv` | — | entire table (12-week baseline = `regular_hours`, no disruption) |

**Bottom line:** the only fields flowing directly from M5 into the factory model are each
product's identity/category and its average selling price, plus its real demand level and
variability (used only to *size* assumed parameters like safety stock, batch sizes, machine routing, and now material structure, never copied in as if they were manufacturing data). Every cost, time, and capacity figure is an explicit, formula-based industrial assumption, kept in this notebook so it can be audited and revised.